In [1]:
%cd ..
%load_ext autoreload
%autoreload 2
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from base64 import b64encode
from cotracker.utils.visualizer import Visualizer, read_video_from_path, read_images_from_path
from IPython.display import HTML

def show_video(video_path):
    video_file = open(video_path, "r+b").read()
    video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
    return HTML(f"""<video width="1280" height="960" autoplay loop controls><source src="{video_url}"></video>""")

/opt/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspaces/co-tracker


In [ ]:
asset_folder = './assets'
img_folder_name = 'walk_869488000/undist_rgb_half_size'
foot_pos_file = "videos/walk_869488000/overlay_depth_cam/foot_pos_for_depth_cam.npy"
base_pos_file = "videos/walk_869488000/overlay_depth_cam/hip_pos.npy"
read_start_frame = 0
read_end_frame = -1
mask_folder_name = None # 'maila/masks'

img_folder_path = os.path.join(asset_folder, img_folder_name)
if mask_folder_name is not None:
    mask_folder_path = os.path.join(asset_folder, mask_folder_name)
else:
    mask_folder_path = None


video = read_images_from_path(img_folder_path, mask_folder_path)[read_start_frame:read_end_frame]
video = torch.from_numpy(video).permute(0, 3, 1, 2)[None].float() # video should be of shape (1, B, C, H, W), with B is number of frames
print(f"Video shape after reshape: {video.shape}")

Found # files: 29
Video shape after reshape: torch.Size([1, 28, 3, 360, 640])


In [9]:


foot_pos = np.load(foot_pos_file)
base_pos = np.load(base_pos_file)[0]

tracks = np.concatenate((foot_pos, base_pos), axis=1)
tracks = tracks.reshape(1, tracks.shape[0], -1, 2)
tracks = torch.tensor(tracks)
tracks.shape

FileNotFoundError: [Errno 2] No such file or directory: 'videos/walk_869488000/overlay_depth_cam/foot_pos_half_res.npy'

In [4]:
import torch
import numpy as np
from scipy import interpolate

def interpolate_tracks(tracks: torch.Tensor) -> torch.Tensor:
    """
    Linearly interpolates 0.0 values in tracks over the time axis.

    Args:
        tracks: Tensor of shape (B, T, N, 2)

    Returns:
        tracks_interp: Tensor of same shape, with 0.0 values interpolated
    """
    tracks_np = tracks.cpu().numpy()  # Convert to numpy for scipy interpolate
    B, T, N, D = tracks_np.shape
    assert D == 2, "Expected last dimension to be 2 (x, y coordinates)."

    for b in range(B):
        for n in range(N):
            for d in range(D):  # x and y
                column = tracks_np[b, :, n, d]  # shape (T,)
                non_zero_mask = column != 0.0
                if non_zero_mask.sum() < 2:
                    # Not enough points to interpolate, skip
                    continue
                x_non_zero = np.where(non_zero_mask)[0]
                y_non_zero = column[non_zero_mask]
                interp_func = interpolate.interp1d(
                    x_non_zero, y_non_zero, bounds_error=False, fill_value="extrapolate"
                )
                column[~non_zero_mask] = interp_func(np.where(~non_zero_mask)[0])
                tracks_np[b, :, n, d] = column

    return torch.from_numpy(tracks_np).to(tracks.device)

tracks = interpolate_tracks(tracks)

In [5]:
video_path = results_path = os.path.join('./videos', img_folder_name.rsplit('.', 1)[0])


vis = Visualizer(
    save_dir=video_path,
    linewidth=3,
    mode='rainbow',
    tracks_leave_trace=-1,
)
vis.visualize(
    video=video,
    tracks=tracks,
    visibility=torch.ones(1, video.shape[1], tracks.shape[2]),
    filename='tracks');

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (640, 360) to (640, 368) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Video saved to ./videos/walk_869488000/undist_rgb_half_size/tracks.mp4
